# Road-trip traffic evaluation — San Diego → Cupertino

A walkthrough of the pipeline: parse GPX → linear-reference onto the route → detect
rare events with rate CIs → cross-validate the two cars as sensors → overlay the drive
on the PeMS speed field → predict-then-validate travel time.

Runs on synthetic data out of the box; point it at real GPX/PeMS to use the real drive.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from src import synth, gpx_io, plots
from src.route import Route, add_route_distance
from src.traces import align_dual, agreement_stats
from src.events import add_accel_jerk, detect_hard_brakes, detect_congestion_onset, summarize_events
from src.pems import synthetic_speed_field
from src.metrics import travel_time_report

# Use real files if present, else generate synthetic ones.
from pathlib import Path
raw = sorted(Path('../data/raw').glob('*.gpx'))
if len(raw) >= 2:
    paths = raw[:2]
else:
    paths = list(synth.generate_pair(out_dir='../data/synthetic'))
paths

## 1. Parse & QC the traces
Speed is derived from consecutive positions; QC reports sampling rate, gaps, and implausible speeds.

In [ ]:
civic = gpx_io.load_trace(paths[0])
cross = gpx_io.load_trace(paths[1])
pd.DataFrame([civic.attrs['qc'], cross.attrs['qc']], index=['civic', 'crosstrek'])

## 2. Linear referencing
Project every point onto a route polyline so both cars share a 'distance along route' axis.

In [ ]:
route = Route.from_trace(civic)
civic = add_route_distance(civic, route)
print(f'route length: {route.length_mi:.1f} mi')

## 3. Rare events + rate estimation
Hard-brake and congestion-onset events, with rates per 100 mi (exact Poisson + bootstrap CIs).

In [ ]:
civic = add_accel_jerk(civic)
brakes = detect_hard_brakes(civic)
congest = detect_congestion_onset(civic)
events = pd.concat([brakes, congest], ignore_index=True)
exposure = float(civic['dist_mi'].iloc[-1])
summarize_events(events, exposure)

In [ ]:
plots.event_map(civic, events)

## 4. Dual-sensor agreement
Two cars, same road → their speed profiles should agree to within GPS noise.

In [ ]:
aligned = align_dual(civic, cross, route)
stats = agreement_stats(aligned)
print(stats)
plots.dual_agreement(aligned, stats)

## 5. The drive on the corridor speed field
Overlay the trajectory on the (here synthetic) PeMS space-time speed field.

In [ ]:
field = synthetic_speed_field(route.length_mi)   # swap for load_pems_5min('../data/pems/us101.csv')
plots.speed_contour(field, civic)

## 6. Predict-then-validate travel time

In [ ]:
travel_time_report(civic, assumed_speed_mph=62.0)

---
**Next:** drop real GPX in `../data/raw/`, export PeMS to `../data/pems/`, and rerun. For Tier 2, calibrate a SUMO segment from PeMS and score it with the GEH statistic in `src/metrics.py`.